In [ ]:
import json
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from rdkit import rdBase, Chem
from sklearn.model_selection import KFold
from tqdm import tqdm

from src import BERT, BERT4IR
from src import load_exps_from_yaml, set_seed, set_default, build_run_id
from src import morgan_tokenize, smiles_tokenize

rdBase.DisableLog('rdApp.warning')

In [ ]:
"""
STEP 1: 指定配置文件 ⚠️
- 在此输入你想要执行推理任务的 YAML 路径。
"""

yaml_file = "configs/finetune/task7_smiles_all_charge_kfold.yaml"

In [ ]:
"""
STEP 2: 环境路径配置
- 获取并切换工作目录，确保相对路径指向项目根目录。
"""

current_dir = Path.cwd()
print(f"当前工作目录：{current_dir}")

# 切换至项目根目录
os.chdir("../../")
root_dir = Path.cwd()
print(f"修改后的根目录：{root_dir}")

# 运行设备
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
"""
STEP 3: 推理辅助工具定义
- get_atom_counts: 统计分子碳原子数及重原子数。
- build_model_from_args: 适配训练逻辑动态构建 BERT4IR 模型。
- build_name: 构建保存文件名。
"""


def get_atom_counts(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 0, 0
    nc = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == 'C')
    n_heavy = mol.GetNumHeavyAtoms()
    return nc, n_heavy


def build_model(args, out_dim):
    c_dim = args.charge_dim
    bert = BERT(args.vocab_size, args.model.hid_dim, args.model.n_layer,
                args.model.n_head, args.model.dropout, args.rope)
    model = BERT4IR(bert, out_dim, args.data.label_normed, args.support_charge, args.charge.vocab,
                    args.charge.enc, c_dim, c_dim, plot=True)
    return model


def build_name(args, scale):
    charge_info = ""
    split_info = ""
    scale_info = ""
    fold_info = ""

    if args.support_charge:
        if args.charge.enc == "emb":
            charge_info = f"{args.charge_dim}"
        if args.charge.enc == "onehot":
            charge_info = f"[{len(args.charge.vocab)}x{args.charge_dim}]"

    if not args.data.split == "random":
        split_info = f"_{args.data.split}"

    if not args.data.scales == [1.0]:
        scale_info = f"_{scale}x"

    if args.data.kfold:
        fold_info = f"_foldx{args.data.kfold}"

    return args.run_id + charge_info + split_info + scale_info + fold_info

In [ ]:
"""
STEP 4: 推理引擎与结果封装
- 将 SMILES 序列化为 Tensor 输入。
- 执行模型前向传播，捕获预测光谱及 EMD 损失。
- 将预测结果与原始元数据封装为字典列表。
"""


def run_inference(dataset_split: Dataset, args, model, vocab, tokenize_fn):
    results = []
    model.eval()

    with torch.no_grad():
        for row in tqdm(dataset_split, desc="推理中", leave=False):
            # 1. 数据预处理
            smi = row["canonical_smiles"]
            label_tensor = torch.tensor(json.loads(row[args.label_col])).unsqueeze(0).to(device)
            charge_tensor = torch.tensor(int(row["charge"])).unsqueeze(0).to(device)

            # 2. 序列分词与编码
            tokens = tokenize_fn(smi)
            if args.encode.scheme == "morgan":  # 针对 Morgan 分词的特殊过滤逻辑
                tokens = tokens[1::2]

            tokens_idx = [vocab.get(t, vocab["<unk>"]) for t in tokens]
            token_ids = [vocab["<cls>"]] + tokens_idx + [vocab["<sep>"]]
            token_ids_tensor = torch.tensor(token_ids[:args.model.seq_len]).unsqueeze(0).to(device)

            # 3. 前向计算
            n_c, n_heavy = get_atom_counts(smi)
            outputs = model(token_ids_tensor, label_tensor, charge_tensor)

            # 4. 结果组装
            res_item = {
                "mid": row['mol_id'],
                "formula": row['formula'],
                "canonical_smiles": smi,
                "n_c": n_c,
                "n_heavy": n_heavy,
                "charge": row['charge'],
                "EMD": outputs[2].item(),
                "normed_y_hat": outputs[0].squeeze().cpu().tolist(),
                "normed_y": outputs[1].squeeze().cpu().tolist()
            }

            results.append(res_item)
    return results

In [ ]:
"""
STEP 5: 核心推理引擎
- 执行单个实验配置下的所有 Fold 推理并汇总结果。
"""


def run_kfold_inference(args, scale):
    save_name = build_name(args, scale)
    all_results = []

    # 1. 准备数据划分 (保持与训练 seed 一致)
    full_data = load_dataset('csv', data_files=args.data.path)['train']
    kf = KFold(n_splits=args.data.kfold, shuffle=True, random_state=args.seed)
    fold_indices = [val_idx for _, val_idx in kf.split(range(len(full_data)))]

    # 2. 配置资源准备
    with open(args.encode.vocab, 'rb') as f:
        vocab = pickle.load(f)
    args.vocab_size = len(vocab)
    tokenize_fn = smiles_tokenize if args.encode.scheme == "smiles" else morgan_tokenize
    out_dim = len(json.loads(full_data[0][args.label_col]))

    # 3. 迭代 K 个 Fold
    for k in range(1, args.data.kfold + 1):
        run_id = build_run_id(args, scale, k)
        weight_path = Path(args.save.model) / args.project.name / "R2" / f"bert4ir_{run_id}.pth"

        if not weight_path.exists():
            print(f"[Skip] Fold {k}: 找不到权重 {weight_path.name}")
            continue

        # 加载模型并推理当前 Fold
        model = build_model(args, out_dim).to(device)
        ckpt = torch.load(weight_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['bert4ir_state_dict'])

        test_subset = full_data.select(fold_indices[k - 1])
        fold_res = run_inference(test_subset, args, model, vocab, tokenize_fn)

        print(f"Fold {k} 平均 EMD: {np.mean([r['EMD'] for r in fold_res]):.4f}")
        all_results.extend(fold_res)

    # 4. 结果持久化
    if all_results:
        res_df = pd.DataFrame(all_results)
        save_path = Path(f"save/results/test_from_bert/{args.project.name}/{save_name}.csv")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        res_df.to_csv(save_path, index=False)
        print(f"✅ 汇总完成: {save_path.name} | 总数: {len(res_df)}")

In [ ]:
"""
STEP 6: 自动化任务分发
- 加载 YAML 配置，自动循环所有参数组合并触发推理引擎。
- 严格拦截：若配置中包含 kfold 参数，则跳过并提示使用专用脚本。
"""

all_exps = load_exps_from_yaml(yaml_file)

for exp_args in all_exps:
    # 1. 拦截包含 kfold 定义的任务
    kfold_val = getattr(exp_args.data, 'kfold', None)

    if kfold_val is None:
        print(f"\n[?] 拦截实验: {exp_args.run_id}")
        print(f"[!]本脚本仅支持 KFold 复现，请使用 [generate_results.ipynb] 文件进行推理。")
        continue

    # 2. 基础环境初始化
    set_seed(getattr(exp_args, "seed", 42))
    exp_args = set_default(exp_args)

    # 3. 展开电荷参数循环
    charge_dim_list = exp_args.charge.onehot_repeat if exp_args.charge.enc == "onehot" else exp_args.charge.emb_dim

    for c_val in charge_dim_list:
        exp_args.charge_dim = c_val

        # 4. 展开数据规模循环
        for scale_value in exp_args.data.scales:
            # 执行推理
            run_kfold_inference(exp_args, scale_value)